# LAB | API Calling to ChatGPT
## Name: Martha Olukayode
## Project Description: 

You've been tasked with building an automation system that:
- Takes a product image and basic metadata (name, price, category)
- Sends it to ChatGPT with a prompt
- Receives a professionally written product listing
- Formats it for use in your e-commerce platform

### Setup

In [24]:
!pip install openai pydantic python-dotenv


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
# **************************************
# Step 1: Setting Up OpenAI API Access
# **************************************

import os
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from typing import List
import random

# Load OPENAI_API_KEY from the .env file next to this notebook
load_dotenv()

# Initialize client (reads OPENAI_API_KEY from the environment)
api_key = os.getenv("OPENAI_API_KEY")

# Check whether Python found the key
if api_key:
    print("✓ API key loaded successfully!")
else:
    print("✗ API key was not found.")

client = OpenAI(api_key=api_key)

print("✓ OpenAI client initialized successfully!")


✓ API key loaded successfully!
✓ OpenAI client initialized successfully!


In [26]:
# **************************************
# Step 2: Preparing the Dataset
# **************************************
from datasets import load_dataset
import requests
from PIL import Image
import pandas as pd
from pathlib import Path

csv_path = Path("data/styles.csv")
images_dir = Path("product_images")

# Check that the CSV exists
if not csv_path.exists():
    raise FileNotFoundError(f"Dataset not found: {csv_path}")

# Load dataset
products_df = pd.read_csv(csv_path)

# Check required columns
required_columns = ["id", "productDisplayName", "subCategory"]

# Load dataset from HuggingFace
print("Loading product dataset...")

missing_columns = [
    col for col in required_columns
    if col not in products_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# Create image directory if needed
images_dir.mkdir(exist_ok=True)

products = []

for _, pro in products_df.iterrows():

    image_path = images_dir / f"{pro['id']}.jpg"

    # Skip products whose image is missing
    if not image_path.exists():
        print(f"⚠ Image missing: {image_path}")
        continue

    product = {
        "id": int(pro["id"]),
        "name": pro["productDisplayName"],
        "price": random.randint(20, 99),
        "category": pro["subCategory"],
        "image_path": str(image_path)
    }

    products.append(product)

print("\n✓ Dataset prepared!")
print(f"Total rows in CSV: {len(products_df)}")
print(f"Products with available images: {len(products)}")

print("\nFirst 3 prepared products:")
for product in products[:3]:
    print(product)




Loading product dataset...

✓ Dataset prepared!
Total rows in CSV: 20
Products with available images: 20

First 3 prepared products:
{'id': 9999, 'name': 'Palm Tree Kids Girl Mm Merlin Top Blue Tops', 'price': 80, 'category': 'Topwear', 'image_path': 'product_images\\9999.jpg'}
{'id': 9998, 'name': 'Palm Tree Girls Hs Livia Top Pink Tops', 'price': 89, 'category': 'Topwear', 'image_path': 'product_images\\9998.jpg'}
{'id': 9997, 'name': 'Palm Tree Girls Hs Livia Top Blue Tops', 'price': 59, 'category': 'Topwear', 'image_path': 'product_images\\9997.jpg'}


In [27]:
# **************************************
# Step 3: Encoding Images for API
# **************************************
import base64
from pathlib import Path

def encode_image_to_base64(image_path):
    """
    Convert a local image file to a Base64 string.

    Args:
        image_path: Valid path to a local image file.

    Returns:
        Base64 encoded image string.
    """
    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(
            f"Image file not found: {image_path}"
        )

    with open(image_path, "rb") as image_file:
        encoded_image = base64.b64encode(image_file.read()).decode('utf-8')

    return encoded_image

# Encode the product image
#img_path = products_df.iloc[0]["image"]
encoded_image = encode_image_to_base64(product["image_path"])

print("\nImage encoded successfully!")
print("\nProduct Information with Encoded Image:")
print("Encoded image length:", len(encoded_image))
print("First 40 characters:", encoded_image[:40] + "...")


Image encoded successfully!

Product Information with Encoded Image:
Encoded image length: 2968
First 40 characters: /9j/4AAQSkZJRgABAQAAZABkAAD/7AARRHVja3kA...


In [28]:
# **************************************
# 4: Creating the Product Listing Prompt
# **************************************
def create_product_listing_prompt(product_name, price, category, additional_info=None):
    """
    Create a prompt for generating product listings.
    
    Parameters:
    - product_name: Name of the product
    - price: Price of the product
    - category: Product category
    - additional_info: Optional additional information
    
    Returns:
    - Formatted prompt string
    """
    prompt = f"""You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.
 
Product Information:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{f'- Additional Info: {additional_info}' if additional_info else ''}
 
Please create a professional product listing that includes:
 
1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive language
   - Include relevant details visible in the image
3. **Key Features** (bullet points, 5-7 items)
4. **SEO Keywords** (comma-separated, 10-15 relevant keywords)
 
Format your response as JSON with the following structure:
{{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2", ...],
    "keywords": "keyword1, keyword2, ..."
}}
 
Be specific about what you see in the image. Mention colors, materials, design elements, and any distinctive features."""
    
    return prompt
 
# Test prompt creation
product_prompt = create_product_listing_prompt(
    product_name="Wireless Bluetooth Headphones",
    price=79.99,
    category="Electronics",
    additional_info="Noise cancelling, 30-hour battery"
)
 
print("\n" + "="*50)
print("PROMPT TEMPLATE")
print("="*50)
print(product_prompt[:500] + "...")  # Show first 500 characters


PROMPT TEMPLATE
You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: Wireless Bluetooth Headphones
- Price: $79.99
- Category: Electronics
- Additional Info: Noise cancelling, 30-hour battery

Please create a professional product listing that includes:

1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive ...


In [29]:
# **************************************
# Step 5: Calling the ChatGPT API with Vision
# **************************************
import os
import base64
import json
from dotenv import load_dotenv
from openai import OpenAI
# Send the product image and prompt to OpenAI
try:
    response = client.responses.create(
        model="gpt-4o-mini",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": product_prompt
                    },
                    {                        
                        "type": "input_image",
                        "image_url": f"data:image/jpeg;base64,{encoded_image}"
                    }
                ]
            }
        ]
    )

    print("\n=== AI GENERATED PRODUCT LISTING ===")
    print(response.output_text)

    # Get the AI response
    response_text = response.output_text.strip()

    # Remove Markdown code blocks if the AI added them
    if response_text.startswith("```json"):
        response_text = response_text[7:]

    if response_text.startswith("```"):
        response_text = response_text[3:]

    if response_text.endswith("```"):
        response_text = response_text[:-3]

    response_text = response_text.strip()

    # Convert JSON text into a Python dictionary
    product_listing = json.loads(response_text)

    print("\n✓ Response converted to Python dictionary!")

    # Save the result to a JSON file
    with open("product_listings.json", "w", encoding="utf-8") as file:
        json.dump(product_listing, file, indent=4, ensure_ascii=False)

    print("✓ Product listing saved to product_listings.json")   

except Exception as e:
    print("API request failed:", e)


=== AI GENERATED PRODUCT LISTING ===
```json
{
    "title": "Premium Noise Cancelling Wireless Bluetooth Headphones",
    "description": "Experience ultimate sound quality and comfort with our Premium Noise Cancelling Wireless Bluetooth Headphones. Perfect for music lovers and busy professionals alike, these headphones feature advanced noise-cancelling technology that immerses you in your favorite tunes without distractions. Enjoy up to 30 hours of uninterrupted listening time on a single charge, making them ideal for long commutes, travel, or home use. Made with soft-touch materials and a sleek, modern design, they are lightweight and comfortable for extended wear. Connect effortlessly to any Bluetooth device and enjoy crystal-clear calls with built-in microphone functionality. Elevate your audio experience today!",
    "features": [
        "Advanced Noise Cancelling Technology",
        "30-Hour Battery Life",
        "Lightweight and Comfortable Design",
        "Crystal-Clear Cal

In [30]:
def generate_product_listing(product):
    """Generate an AI product listing for one product."""

    try:
        print(f"\nProcessing: {product['name']}...")

        # 1. Encode the product image
        encoded_image = encode_image_to_base64(
            product["image_path"]
        )

        # 2. Create the prompt
        product_prompt = create_product_listing_prompt(
            product["name"],
            product["price"],
            product["category"]
        )

        # 3. Send prompt and image to OpenAI
        response = client.responses.create(
            model="gpt-4o-mini",
            input=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "input_text",
                            "text": product_prompt
                        },
                        {
                            "type": "input_image",
                            "image_url": (
                                f"data:image/jpeg;base64,"
                                f"{encoded_image}"
                            )
                        }
                    ]
                }
            ]
        )

        # 4. Get the response text
        response_text = response.output_text.strip()

        # 5. Remove Markdown formatting if present
        if response_text.startswith("```json"):
            response_text = response_text[7:]

        if response_text.startswith("```"):
            response_text = response_text[3:]

        if response_text.endswith("```"):
            response_text = response_text[:-3]

        response_text = response_text.strip()

        # 6. Convert JSON text into a Python dictionary
        listing = json.loads(response_text)

        print(f"✓ Listing generated for {product['name']}")

        return listing

    except Exception as e:
        print(f"✗ Error processing {product['name']}: {e}")
        return None

In [33]:
# ****************************************
# Step 6: Processing Multiple Products
# ****************************************

import time

# Store successful and failed results
all_listings = []
failed_products = []

# Required fields expected from the AI response
required_fields = ["title", "description", "features", "keywords"]

# Process each product
for product in products:

    print(f"\nProcessing product ID: {product['id']}")

    listing = generate_product_listing(product)

    if listing is not None:

        # Validate the generated JSON
        missing_fields = [
            field for field in required_fields
            if field not in listing
        ]

        if missing_fields:
            print(
                f"✗ Product {product['id']} missing fields: "
                f"{missing_fields}"
            )

            failed_products.append({
                "id": product["id"],
                "reason": f"Missing fields: {missing_fields}"
            })

        else:
            # Add original product information
            listing["product_id"] = product["id"]
            listing["original_name"] = product["name"]
            listing["price"] = product["price"]
            listing["category"] = product["category"]

            all_listings.append(listing)

            print(f"✓ Product {product['id']} completed successfully")

    else:
        # Keep track of failed products
        failed_products.append({
            "id": product["id"],
            "reason": "Listing generation failed"
        })

    # Small delay between API requests
    time.sleep(1)


# Save successful listings
with open(
    "product_listings.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        all_listings,
        file,
        indent=4,
        ensure_ascii=False
    )


# Save failed products separately
with open(
    "failed_products.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        failed_products,
        file,
        indent=4,
        ensure_ascii=False
    )


# Batch summary
print("\n=================================")
print("PROCESSING COMPLETE")
print("=================================")

print(f"Total products: {len(products)}")
print(f"Successful listings: {len(all_listings)}")
print(f"Failed products: {len(failed_products)}")

print("\nResults saved to product_listings.json")
print("Failures saved to failed_products.json")


Processing product ID: 9999

Processing: Palm Tree Kids Girl Mm Merlin Top Blue Tops...
✓ Listing generated for Palm Tree Kids Girl Mm Merlin Top Blue Tops
✓ Product 9999 completed successfully

Processing product ID: 9998

Processing: Palm Tree Girls Hs Livia Top Pink Tops...
✓ Listing generated for Palm Tree Girls Hs Livia Top Pink Tops
✓ Product 9998 completed successfully

Processing product ID: 9997

Processing: Palm Tree Girls Hs Livia Top Blue Tops...
✓ Listing generated for Palm Tree Girls Hs Livia Top Blue Tops
✓ Product 9997 completed successfully

Processing product ID: 9996

Processing: Palm Tree Girls Sp Jelly Top Blue Tops...
✓ Listing generated for Palm Tree Girls Sp Jelly Top Blue Tops
✓ Product 9996 completed successfully

Processing product ID: 9995

Processing: Gini And Jony Girls Su Scotia Sko Pink Skirts...
✓ Listing generated for Gini And Jony Girls Su Scotia Sko Pink Skirts
✓ Product 9995 completed successfully

Processing product ID: 9994

Processing: Gini And 